<a href="https://colab.research.google.com/github/aravindh28/swiftcounter-cv/blob/dev%2Fam2/Pipeline_v3_cleanup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab Setup

In [ ]:

# -*- coding: utf-8 -*-
"""
Cleaned Bird Classification Pipeline
Optimized for multiple bird counting with simplified, efficient code
"""

# ============================================================================
# COLAB SETUP
# ============================================================================

# Install required packages
!pip install yt-dlp ultralytics opencv-python tqdm numpy

# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

# ============================================================================
# IMPORTS AND SETUP
# ============================================================================

import numpy as np
import cv2
import torch
import json
import math
from tqdm import tqdm
from ultralytics import YOLO, SAM
from datetime import datetime
import os


Config


In [ ]:


# ============================================================================
# CONFIGURATION
# ============================================================================

# Set your path - UPDATE THIS to match your Google Drive structure
PATH = "drive/MyDrive/UW-MS-DS/CSE493G1/Final_Project/Github/swiftcounter-cv/experiments"

# Check if path exists and create data directory if needed
!ls {PATH}
!mkdir -p {PATH}/data

print(f"✅ Working directory: {PATH}")
print(f"📁 Data directory: {PATH}/data")

# IMPORTANT: Ensure cookies.txt is in the data directory for YouTube downloads
print("📌 IMPORTANT: Make sure cookies.txt is in the data directory for video downloads!")
print(f"   Expected location: {PATH}/data/cookies.txt")

# ROI polygon (rectangle around chimney)
roi_polygon = np.array([
    [580, 550],  # top-left
    [780, 550],  # top-right
    [780, 720],  # bottom-right
    [580, 720],  # bottom-left
], dtype=int)

# Save ROI for use by tracker
roi_save_path = f"{PATH}/data/roi_polygon.npy"
np.save(roi_save_path, roi_polygon)
print(f"✅ ROI coordinates: {roi_polygon.tolist()}")
print(f"✅ ROI saved to: {roi_save_path}")

# Test segments configuration
segments_to_test = [
    {"start": "0:05", "end": "0:16", "expected_count": 47, "name": "original_working"}
]

VIDEO_URL = "https://www.youtube.com/watch?v=mkquDCpPD9c

Helper functions


In [ ]:
# ============================================================================
# HELPER FUNCTIONS (Including Integrated Video Download)
# ============================================================================

import subprocess
import yt_dlp

def time_str_to_seconds(time_str):
    """Convert MM:SS format to seconds for yt-dlp"""
    if ':' in time_str:
        parts = time_str.split(':')
        if len(parts) == 2:
            minutes, seconds = parts
            return int(minutes) * 60 + int(seconds)
        elif len(parts) == 3:
            hours, minutes, seconds = parts
            return int(hours) * 3600 + int(minutes) * 60 + int(seconds)
    return int(time_str)

def time_str_to_filename(time_str):
    """Convert MM:SS to filename-safe format"""
    return time_str.replace(':', '-')

def precise_trim(input_path, output_path, start, end):
    """Precise video trimming using ffmpeg for frame accuracy"""
    duration = end - start
    print(f"✂️  Trimming {input_path} from {start}s to {end}s into {output_path}")
    cmd = [
        "ffmpeg",
        "-y",
        "-i", input_path,
        "-ss", str(start),
        "-t", str(duration),
        "-c:v", "libx264",
        "-an",        # no audio
        output_path
    ]
    result = subprocess.run(cmd, capture_output=True)
    if result.returncode == 0:
        print("✅ Precise trim complete!")
        return True
    else:
        print(f"❌ ffmpeg trimming failed:\n{result.stderr.decode()}")
        return False

def download_video(url, filename, start=None, end=None):
    """Integrated video download using yt-dlp with optional precise trimming"""
    # Download to a temporary file if segmenting
    temp_filename = filename
    if start is not None and end is not None:
        temp_filename = filename.replace('.mp4', '_raw.mp4')

    if os.path.exists(filename) and os.path.getsize(filename) > 0:
        print(f"✅ File '{filename}' already exists and is non-empty. Skipping download.")
        return True

    ydl_opts = {
        "outtmpl": temp_filename,
        "format": "bestvideo[height<=720][ext=mp4]/bestvideo[height<=720]/best",
        "merge_output_format": "mp4",
        "quiet": False,
        "noplaylist": True,
    }

    # Use cookies if found in data/
    cookies_path = os.path.join(f"{PATH}/data", "cookies.txt")
    if os.path.exists(cookies_path):
        print(f"🔐 Using cookies from '{cookies_path}'")
        ydl_opts["cookies"] = cookies_path

    print(f"📥 Downloading video from {url} to {temp_filename}")
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
            print("✅ Download complete!")
    except Exception as e:
        print(f"❌ Download failed:\n{e}")
        return False

    # If segment requested, trim with ffmpeg for frame accuracy
    if start is not None and end is not None:
        success = precise_trim(temp_filename, filename, start, end)
        if temp_filename != filename and os.path.exists(temp_filename):
            os.remove(temp_filename)  # Clean up temp file
        return success

    return True

def download_video_segment_unique(video_url, start_time, end_time, segment_name):
    """Download video segment with caching - only downloads if file doesn't exist"""
    start_sec = time_str_to_seconds(start_time)
    end_sec = time_str_to_seconds(end_time)

    # Create unique filename
    start_filename = time_str_to_filename(start_time)
    end_filename = time_str_to_filename(end_time)
    unique_filename = f"downloaded_video_{start_filename}_to_{end_filename}.mp4"
    unique_path = f"{PATH}/data/{unique_filename}"

    # CHECK IF FILE ALREADY EXISTS (CACHING FIX)
    if os.path.exists(unique_path) and os.path.getsize(unique_path) > 0:
        print(f"✅ Using existing file: {unique_filename}")
        return unique_path, unique_filename

    print(f"📥 Downloading segment '{segment_name}': {start_time} to {end_time}")
    print(f"   Time range: {start_sec}s to {end_sec}s")
    print(f"   Saving as: {unique_filename}")

    # Ensure the data directory exists
    os.makedirs(f"{PATH}/data", exist_ok=True)

    # Use integrated download function instead of external script
    success = download_video(video_url, unique_path, start_sec, end_sec)

    if success and os.path.exists(unique_path):
        print(f"✅ Video saved as: {unique_filename}")
        return unique_path, unique_filename
    else:
        print(f"❌ Failed to download video segment")
        return None, None

Bird tracker class


In [ ]:


# ============================================================================
# SIMPLIFIED BIRD TRACKER CLASS
# ============================================================================

class AggressiveBirdTracker:
    """Simplified bird tracker focused on multiple bird detection"""

    def __init__(self, roi_polygon,
                 # Core parameters only
                 yolo_confidence=0.08,
                 roi_entry_threshold=0.1,
                 tracking_zone_radius=150,
                 min_track_length=3,
                 max_distance_for_matching=80,

                 # Recovery system
                 track_recovery_enabled=True,
                 max_frames_to_recover=5,
                 recovery_distance_threshold=120,
                 recovery_confidence_threshold=0.6,

                 # Debug
                 debug_mode=True):

        # Core setup
        self.roi_polygon = roi_polygon
        self.bird_tracks = {}
        self.next_track_id = 1
        self.birds_entered_roi = set()
        self.frame_count = 0

        # Parameters
        self.yolo_confidence = yolo_confidence
        self.roi_entry_threshold = roi_entry_threshold
        self.tracking_zone_radius = tracking_zone_radius
        self.min_track_length = min_track_length
        self.max_distance_for_matching = max_distance_for_matching

        # Recovery system
        self.track_recovery_enabled = track_recovery_enabled
        self.max_frames_to_recover = max_frames_to_recover
        self.recovery_distance_threshold = recovery_distance_threshold
        self.recovery_confidence_threshold = recovery_confidence_threshold

        self.debug_mode = debug_mode

        # ROI bounds calculation
        self.roi_min_x, self.roi_min_y = np.min(self.roi_polygon, axis=0)
        self.roi_max_x, self.roi_max_y = np.max(self.roi_polygon, axis=0)
        self.roi_center = ((self.roi_min_x + self.roi_max_x) // 2,
                          (self.roi_min_y + self.roi_max_y) // 2)

        # Recovery system
        self.recently_lost_tracks = {}
        self.recovery_stats = {'recoveries_attempted': 0, 'recoveries_successful': 0}

        print(f"🎯 Bird Tracker initialized for multiple bird detection:")
        print(f"   YOLO confidence: {self.yolo_confidence}")
        print(f"   ROI rectangle: ({self.roi_min_x},{self.roi_min_y}) to ({self.roi_max_x},{self.roi_max_y})")
        print(f"   Tracking zone: {self.tracking_zone_radius}px radius")
        print(f"   Track Recovery: {'ENABLED' if self.track_recovery_enabled else 'DISABLED'}")

    def get_mask_centroid(self, mask):
        """Calculate centroid of segmentation mask"""
        y_coords, x_coords = np.where(mask > 0)
        if len(x_coords) == 0:
            return None
        return (int(np.mean(x_coords)), int(np.mean(y_coords)))

    def calculate_distance(self, point1, point2):
        """Calculate Euclidean distance between two points"""
        return math.sqrt((point1[0] - point2[0])**2 + (point1[1] - point2[1])**2)

    def is_point_in_roi_rectangle(self, point):
        """Check if point is inside ROI rectangle"""
        x, y = point
        return (self.roi_min_x <= x <= self.roi_max_x and
                self.roi_min_y <= y <= self.roi_max_y)

    def is_in_tracking_zone(self, centroid):
        """Check if bird is within tracking zone"""
        distance_to_center = self.calculate_distance(centroid, self.roi_center)
        return distance_to_center <= self.tracking_zone_radius

    def simple_roi_detection(self, bird_mask):
        """Simplified ROI detection for multiple birds"""
        try:
            y_coords, x_coords = np.where(bird_mask > 0)
            if len(x_coords) == 0:
                return False, 0.0

            # Check if any pixels overlap with ROI rectangle
            roi_pixels = 0
            for i in range(0, len(x_coords), 2):  # Sample every 2nd pixel for speed
                x, y = x_coords[i], y_coords[i]
                if self.is_point_in_roi_rectangle((x, y)):
                    roi_pixels += 1

            if roi_pixels > 0:
                return True, 1.0

            # Check centroid with small buffer
            centroid_x = int(np.mean(x_coords))
            centroid_y = int(np.mean(y_coords))

            buffer = 5
            for dx in range(-buffer, buffer + 1):
                for dy in range(-buffer, buffer + 1):
                    test_point = (centroid_x + dx, centroid_y + dy)
                    if self.is_point_in_roi_rectangle(test_point):
                        return True, 0.8

            return False, 0.0

        except Exception as e:
            if self.debug_mode:
                print(f"ROI detection error: {e}")
            return False, 0.0

    def get_sam2_mask_id(self, mask, mask_index=None):
        """Generate stable mask ID"""
        if mask_index is not None:
            return f"sam2_mask_{mask_index}"
        return None

    def predict_next_position(self, track_data):
        """Simple position prediction for recovery"""
        centroid_history = track_data.get('centroid_history', [])

        if len(centroid_history) < 2:
            return centroid_history[-1] if centroid_history else None

        # Linear prediction from last 2 positions
        last_pos = centroid_history[-1]
        prev_pos = centroid_history[-2]

        vx = last_pos[0] - prev_pos[0]
        vy = last_pos[1] - prev_pos[1]

        predicted_pos = (int(last_pos[0] + vx), int(last_pos[1] + vy))
        return predicted_pos

    def attempt_track_recovery(self, unmatched_detections):
        """Track recovery system for handling multiple birds"""
        if not self.track_recovery_enabled or not self.recently_lost_tracks:
            return unmatched_detections, []

        recovered_tracks = []
        still_unmatched = []

        for detection in unmatched_detections:
            detection_centroid = detection['centroid']
            best_recovery_score = 0.0
            best_recovery_track_id = None

            for lost_track_id, lost_info in self.recently_lost_tracks.items():
                frames_since_lost = self.frame_count - lost_info['lost_frame']

                if frames_since_lost > self.max_frames_to_recover:
                    continue

                predicted_pos = self.predict_next_position(lost_info['track_data'])
                if predicted_pos:
                    distance = self.calculate_distance(detection_centroid, predicted_pos)

                    if distance <= self.recovery_distance_threshold:
                        score = 1.0 - (distance / self.recovery_distance_threshold)
                        score *= (1.0 - (frames_since_lost / self.max_frames_to_recover))

                        if score > best_recovery_score and score >= self.recovery_confidence_threshold:
                            best_recovery_score = score
                            best_recovery_track_id = lost_track_id

            if best_recovery_track_id is not None:
                # Recover the track
                lost_info = self.recently_lost_tracks[best_recovery_track_id]
                recovered_track_data = lost_info['track_data'].copy()

                recovered_track_data['centroid_history'].append(detection_centroid)
                recovered_track_data['last_seen'] = self.frame_count
                recovered_track_data['sam2_mask_id'] = detection.get('sam2_mask_id')

                self.bird_tracks[best_recovery_track_id] = recovered_track_data
                del self.recently_lost_tracks[best_recovery_track_id]

                self.recovery_stats['recoveries_attempted'] += 1
                self.recovery_stats['recoveries_successful'] += 1

                recovered_tracks.append({
                    'track_id': best_recovery_track_id,
                    'detection': detection
                })

                if self.debug_mode:
                    gap_frames = self.frame_count - lost_info['lost_frame']
                    print(f"🔄 RECOVERED Track #{best_recovery_track_id} after {gap_frames} frames")
            else:
                still_unmatched.append(detection)

        return still_unmatched, recovered_tracks

    def update_recently_lost_tracks(self):
        """Update recently lost tracks buffer"""
        tracks_to_move = []
        for track_id, track_data in self.bird_tracks.items():
            frames_since_seen = self.frame_count - track_data['last_seen']
            if frames_since_seen >= 1:
                tracks_to_move.append(track_id)

        for track_id in tracks_to_move:
            if track_id not in self.recently_lost_tracks:
                self.recently_lost_tracks[track_id] = {
                    'track_data': self.bird_tracks[track_id].copy(),
                    'lost_frame': self.bird_tracks[track_id]['last_seen'] + 1
                }
            del self.bird_tracks[track_id]

        # Clean up old tracks
        old_tracks = []
        for track_id, lost_info in self.recently_lost_tracks.items():
            frames_since_lost = self.frame_count - lost_info['lost_frame']
            if frames_since_lost > self.max_frames_to_recover:
                old_tracks.append(track_id)

        for track_id in old_tracks:
            del self.recently_lost_tracks[track_id]

    def update_tracks(self, bird_detections):
        """Main tracking update for multiple birds"""
        self.frame_count += 1
        self.update_recently_lost_tracks()

        # Filter detections to tracking zone
        zone_detections = []
        birds_in_roi_this_frame = 0

        for i, detection in enumerate(bird_detections):
            mask = detection['mask']
            confidence = float(detection['confidence'])
            sam2_mask_id = detection.get('sam2_mask_id', self.get_sam2_mask_id(mask, mask_index=i))
            centroid = self.get_mask_centroid(mask)

            if centroid and self.is_in_tracking_zone(centroid):
                in_roi, roi_confidence = self.simple_roi_detection(mask)
                if in_roi:
                    birds_in_roi_this_frame += 1

                zone_detections.append({
                    'centroid': centroid,
                    'mask': mask,
                    'confidence': confidence,
                    'roi_confidence': roi_confidence,
                    'in_roi': in_roi,
                    'sam2_mask_id': sam2_mask_id
                })

        # Track matching
        matched_tracks = set()
        unmatched_detections = []

        for curr_data in zone_detections:
            curr_centroid = curr_data['centroid']
            sam2_mask_id = curr_data['sam2_mask_id']
            best_match_id = None
            best_score = 0.0

            # Try SAM2 ID matching first
            for track_id, track_data in self.bird_tracks.items():
                if track_id in matched_tracks:
                    continue

                if (track_data.get('sam2_mask_id') == sam2_mask_id and
                    sam2_mask_id is not None):
                    best_match_id = track_id
                    best_score = 1.0
                    break

            # Distance matching as fallback
            if best_match_id is None:
                for track_id, track_data in self.bird_tracks.items():
                    if track_id in matched_tracks:
                        continue

                    last_centroid = track_data['centroid_history'][-1]
                    distance = self.calculate_distance(curr_centroid, last_centroid)

                    if distance <= self.max_distance_for_matching:
                        score = 1.0 - (distance / self.max_distance_for_matching)
                        if score > best_score:
                            best_score = score
                            best_match_id = track_id

            if best_match_id is not None and best_score > 0.3:
                # Update existing track
                matched_tracks.add(best_match_id)
                track_data = self.bird_tracks[best_match_id]
                track_data['centroid_history'].append(curr_centroid)
                track_data['last_seen'] = self.frame_count
                track_data['sam2_mask_id'] = sam2_mask_id

                # Check for ROI entry
                if not track_data['entered_roi']:
                    roi_entered = (curr_data['in_roi'] and
                                  curr_data['roi_confidence'] >= self.roi_entry_threshold and
                                  len(track_data['centroid_history']) >= self.min_track_length)

                    if roi_entered:
                        track_data['entered_roi'] = True
                        track_data['entry_location'] = curr_centroid
                        track_data['entry_frame'] = self.frame_count
                        self.birds_entered_roi.add(best_match_id)

                        print(f"🐦 Bird #{best_match_id} entered ROI at frame {self.frame_count}")
                        print(f"   Entry location: {curr_centroid}")
                        print(f"   Track length: {len(track_data['centroid_history'])} frames")
            else:
                unmatched_detections.append(curr_data)

        # Track recovery
        still_unmatched, recovered_tracks = self.attempt_track_recovery(unmatched_detections)

        # Process recovered tracks for ROI entry
        for recovery_info in recovered_tracks:
            track_id = recovery_info['track_id']
            detection = recovery_info['detection']
            track_data = self.bird_tracks[track_id]

            if not track_data['entered_roi']:
                roi_entered = (detection['in_roi'] and
                              detection['roi_confidence'] >= self.roi_entry_threshold and
                              len(track_data['centroid_history']) >= self.min_track_length)

                if roi_entered:
                    track_data['entered_roi'] = True
                    track_data['entry_location'] = detection['centroid']
                    track_data['entry_frame'] = self.frame_count
                    self.birds_entered_roi.add(track_id)
                    print(f"🐦 RECOVERED Bird #{track_id} entered ROI at frame {self.frame_count}")

        # Create new tracks
        for new_data in still_unmatched:
            track_id = self.next_track_id

            self.bird_tracks[track_id] = {
                'centroid_history': [new_data['centroid']],
                'entered_roi': False,
                'last_seen': self.frame_count,
                'sam2_mask_id': new_data['sam2_mask_id']
            }

            # Immediate ROI entry for new tracks (if min_track_length is 1)
            if (new_data['in_roi'] and
                new_data['roi_confidence'] >= self.roi_entry_threshold and
                self.min_track_length <= 1):

                self.bird_tracks[track_id]['entered_roi'] = True
                self.bird_tracks[track_id]['entry_location'] = new_data['centroid']
                self.bird_tracks[track_id]['entry_frame'] = self.frame_count
                self.birds_entered_roi.add(track_id)
                print(f"🐦 NEW Bird #{track_id} entered ROI at frame {self.frame_count}")

            self.next_track_id += 1

        # Debug output for multiple birds
        if birds_in_roi_this_frame > 0 or len(zone_detections) > 0:
            active_tracks = len(self.bird_tracks)
            lost_tracks = len(self.recently_lost_tracks)
            counted_birds = len(self.birds_entered_roi)

            print(f"🔍 Frame {self.frame_count}: {len(zone_detections)} detections, "
                  f"{birds_in_roi_this_frame} in ROI, {active_tracks} active, "
                  f"{lost_tracks} lost, {counted_birds} counted")

    def process_end_of_video(self):
        """End-of-video processing for remaining tracks"""
        print(f"\n🏁 End-of-video processing - checking {len(self.bird_tracks)} tracks...")

        final_counts_added = 0
        for track_id, track_data in self.bird_tracks.items():
            if not track_data['entered_roi']:
                # Check recent frames for ROI presence
                recent_frames = min(3, len(track_data['centroid_history']))
                roi_hits = 0

                for i in range(-recent_frames, 0):
                    if abs(i) <= len(track_data['centroid_history']):
                        centroid_idx = len(track_data['centroid_history']) + i
                        if centroid_idx >= 0:
                            centroid = track_data['centroid_history'][centroid_idx]
                            if self.is_point_in_roi_rectangle(centroid):
                                roi_hits += 1

                if roi_hits >= 1:  # Any ROI hit in recent frames
                    track_data['entered_roi'] = True
                    track_data['entry_location'] = track_data['centroid_history'][-1]
                    track_data['entry_frame'] = self.frame_count
                    self.birds_entered_roi.add(track_id)
                    final_counts_added += 1
                    print(f"🐦 FINAL: Bird #{track_id} counted (was in ROI {roi_hits}/{recent_frames} recent frames)")

        if final_counts_added > 0:
            print(f"✅ Added {final_counts_added} birds from end-of-video processing")

        return final_counts_added

    def get_count(self):
        """Get total count of birds that entered ROI"""
        return len(self.birds_entered_roi)

    def get_tracking_info(self):
        """Get tracking information"""
        return {
            'total_birds_entered': len(self.birds_entered_roi),
            'active_tracks': len(self.bird_tracks),
            'recently_lost_tracks': len(self.recently_lost_tracks),
            'confirmed_entries': list(self.birds_entered_roi),
            'recovery_stats': self.recovery_stats.copy()
        }

Detection and processing

In [ ]:


# ============================================================================
# DETECTION AND PROCESSING FUNCTIONS
# ============================================================================

def conservative_yolo_detection(model, frame, device, confidence=0.08, iou=0.4, max_det=100):
    """YOLO detection with optimized parameters"""
    results = model.predict(
        source=frame,
        device=device,
        conf=confidence,
        iou=iou,
        max_det=max_det,
        verbose=False
    )[0]
    return results

def aggressive_bird_counting_pipeline(video_path, output_suffix=""):
    """Main processing pipeline optimized for multiple birds"""

    # Generate output names
    base_name = os.path.splitext(os.path.basename(video_path))[0]
    output_video_path = f"{PATH}/data/sam2_output_{base_name}{output_suffix}.mp4"
    output_json_path = f"{PATH}/data/sam2_results_{base_name}{output_suffix}.json"

    # Load ROI from saved file
    roi_path = f"{PATH}/data/roi_polygon.npy"
    roi_polygon = np.load(roi_path)

    # Initialize tracker
    bird_tracker = AggressiveBirdTracker(roi_polygon)

    # Load models
    if not os.path.exists("yolo11x.pt"):
        print("📥 Downloading YOLO model...")
        import subprocess
        subprocess.run(["wget", "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11x.pt"],
                      check=True, capture_output=True)

    yolo = YOLO("yolo11x.pt")
    sam = SAM("sam2_b.pt")
    device = 0 if torch.cuda.is_available() else "cpu"

    # Video setup
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    print(f"🎯 PROCESSING FOR MULTIPLE BIRDS")
    print(f"🎯 Video info: {frame_count} total frames at {fps:.1f} fps")

    # Main processing loop
    all_results = []
    frame_idx = 0

    with tqdm(total=frame_count, desc="Processing multiple birds") as pbar:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            try:
                vis_frame = frame.copy()

                # Draw ROI
                cv2.polylines(vis_frame, [roi_polygon], isClosed=True, color=(0, 255, 255), thickness=2)
                cv2.putText(vis_frame, "CHIMNEY ROI", (roi_polygon[0][0], roi_polygon[0][1] - 10),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

                # YOLO detection
                yolo_result = conservative_yolo_detection(yolo, frame, device)

                # Filter for birds
                bird_boxes = []
                bird_confidences = []

                if yolo_result and yolo_result.boxes is not None:
                    for box in yolo_result.boxes:
                        try:
                            cls_id = int(box.cls)
                            confidence = float(box.conf.item()) if hasattr(box.conf, 'item') else float(box.conf)
                            class_name = yolo.names[cls_id]

                            if class_name == 'bird' and confidence >= 0.05:
                                bird_boxes.append(box.xyxy[0].cpu().numpy().tolist())
                                bird_confidences.append(confidence)
                        except:
                            continue

                bird_detections = []

                if bird_boxes:
                    try:
                        # SAM segmentation
                        sam_result = sam.predict(source=frame, bboxes=bird_boxes, device=device, verbose=False)[0]

                        if sam_result.masks is not None:
                            masks = sam_result.masks.data.cpu().numpy()

                            for i, mask in enumerate(masks):
                                try:
                                    confidence = bird_confidences[i]
                                    sam2_mask_id = bird_tracker.get_sam2_mask_id(mask, mask_index=i)

                                    bird_detections.append({
                                        'mask': mask,
                                        'confidence': confidence,
                                        'bbox': bird_boxes[i],
                                        'frame_idx': frame_idx,
                                        'sam2_mask_id': sam2_mask_id
                                    })

                                    # Visualization
                                    color = [int(c) for c in np.random.randint(80, 220, 3)]
                                    mask_img = (mask * 255).astype(np.uint8)
                                    contours, _ = cv2.findContours(mask_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

                                    if contours:
                                        cv2.drawContours(vis_frame, contours, -1, color, 2)
                                        centroid = bird_tracker.get_mask_centroid(mask)
                                        if centroid and 0 <= centroid[0] < width and 0 <= centroid[1] < height:
                                            cv2.putText(vis_frame, f"{confidence:.2f}",
                                                       (centroid[0] + 5, centroid[1] - 5),
                                                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
                                except:
                                    continue
                    except Exception as e:
                        print(f"SAM error at frame {frame_idx}: {e}")

                # Update tracking
                bird_tracker.update_tracks(bird_detections)

                # Get current count
                tracking_info = bird_tracker.get_tracking_info()
                current_count = tracking_info['total_birds_entered']

                # Text overlay
                overlay = vis_frame.copy()
                cv2.rectangle(overlay, (5, 5), (400, 120), (0, 0, 0), -1)
                vis_frame = cv2.addWeighted(vis_frame, 0.8, overlay, 0.2, 0)

                cv2.putText(vis_frame, f"BIRDS COUNTED: {current_count}",
                           (10, 35), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)

                cv2.putText(vis_frame, f"Frame: {frame_idx:03d}/{frame_count} | Detected: {len(bird_detections):02d}",
                           (10, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)

                cv2.putText(vis_frame, f"Active: {tracking_info['active_tracks']:02d} | Lost: {tracking_info['recently_lost_tracks']:02d}",
                           (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)

                # Show counted birds as green dots
                for track_id, track_data in bird_tracker.bird_tracks.items():
                    if track_data['entered_roi'] and 'entry_location' in track_data:
                        entry_point = track_data['entry_location']
                        if (entry_point and len(entry_point) == 2 and
                            0 <= entry_point[0] < width and 0 <= entry_point[1] < height):
                            cv2.circle(vis_frame, tuple(entry_point), 3, (0, 255, 0), -1)
                            cv2.putText(vis_frame, f"#{track_id}",
                                       (entry_point[0] + 6, entry_point[1] - 3),
                                       cv2.FONT_HERSHEY_SIMPLEX, 0.3, (0, 255, 0), 1)

                # Save frame data
                frame_results = {
                    'frame_idx': frame_idx,
                    'total_count': current_count,
                    'birds_detected': len(bird_detections),
                    'active_tracks': tracking_info['active_tracks']
                }
                all_results.append(frame_results)

                out.write(vis_frame)
                frame_idx += 1
                pbar.update(1)

            except Exception as e:
                print(f"Error at frame {frame_idx}: {e}")
                frame_idx += 1
                pbar.update(1)
                continue

    cap.release()
    out.release()

    # End processing
    final_birds_added = bird_tracker.process_end_of_video()
    final_count = bird_tracker.get_count()

    # Results
    final_results = {
        'summary': {
            'video_file': os.path.basename(video_path),
            'total_frames_processed': frame_idx,
            'total_birds_counted': final_count,
            'birds_added_at_end': final_birds_added,
            'method': 'simplified_multiple_bird_tracking'
        },
        'frame_by_frame': all_results,
        'roi_polygon': roi_polygon.tolist()
    }

    with open(output_json_path, "w") as f:
        json.dump(final_results, f, indent=4)

    print(f"\n✅ PROCESSING COMPLETE!")
    print(f"🐦 FINAL COUNT: {final_count} birds")
    print(f"📊 End-of-video adds: +{final_birds_added} birds")
    print(f"📹 Output video: {output_video_path}")
    print(f"📊 Results JSON: {output_json_path}")

    return final_results

In [ ]:


# ============================================================================
# MAIN TESTING FUNCTIONS
# ============================================================================

def run_segment_test(segment_info, video_url):
    """Run tracker on a single segment"""
    segment_name = segment_info['name']
    start_time = segment_info['start']
    end_time = segment_info['end']
    expected_count = segment_info.get('expected_count', None)

    print(f"\n🔄 TESTING SEGMENT: {segment_name}")
    print(f"⏱️ Time range: {start_time} to {end_time}")
    print("-" * 50)

    try:
        # Download video segment (with caching)
        video_path, video_filename = download_video_segment_unique(
            video_url, start_time, end_time, segment_name
        )

        if not os.path.exists(video_path):
            raise Exception(f"Video file not found: {video_path}")

        # Run processing pipeline
        print(f"🤖 Running tracker on: {video_filename}")
        results = aggressive_bird_counting_pipeline(video_path, output_suffix=f"_{segment_name}")

        actual_count = results['summary']['total_birds_counted']

        # Calculate accuracy
        accuracy_info = {}
        if expected_count:
            accuracy_pct = (actual_count / expected_count) * 100
            difference = actual_count - expected_count
            accuracy_info = {
                'expected': expected_count,
                'actual': actual_count,
                'accuracy_percentage': accuracy_pct,
                'difference': difference,
                'status': 'EXCELLENT' if accuracy_pct >= 90 else 'GOOD' if accuracy_pct >= 80 else 'NEEDS_WORK'
            }

        return {
            'segment_name': segment_name,
            'time_range': f"{start_time} to {end_time}",
            'video_filename': video_filename,
            'model_count': actual_count,
            'accuracy_info': accuracy_info,
            'full_results': results,
            'success': True
        }

    except Exception as e:
        print(f"❌ Error in segment {segment_name}: {e}")
        return {
            'segment_name': segment_name,
            'time_range': f"{start_time} to {end_time}",
            'video_filename': 'N/A',
            'model_count': 0,
            'accuracy_info': {'status': 'ERROR'},
            'error': str(e),
            'success': False
        }

def display_comprehensive_summary(all_results):
    """Display detailed summary of all test results"""
    print(f"\n" + "="*85)
    print(f"📊 COMPREHENSIVE TEST RESULTS SUMMARY")
    print(f"="*85)

    print(f"{'Segment':<18} {'Time Range':<15} {'Expected':<10} {'Actual':<10} {'Accuracy':<12} {'Status':<15}")
    print(f"{'-'*18} {'-'*15} {'-'*10} {'-'*10} {'-'*12} {'-'*15}")

    total_expected = 0
    total_actual = 0
    successful_tests = 0

    for result in all_results:
        segment_name = result['segment_name'][:17]
        time_range = result['time_range'][:14]
        model_count = result['model_count']

        if result['success'] and result['accuracy_info'] and 'expected' in result['accuracy_info']:
            expected = result['accuracy_info']['expected']
            accuracy = f"{result['accuracy_info']['accuracy_percentage']:.1f}%"
            status = result['accuracy_info']['status']

            total_expected += expected
            total_actual += model_count
            successful_tests += 1

            print(f"{segment_name:<18} {time_range:<15} {expected:<10} {model_count:<10} {accuracy:<12} {status:<15}")
        else:
            status = "ERROR" if not result['success'] else "MODEL_ONLY"
            print(f"{segment_name:<18} {time_range:<15} {'N/A':<10} {model_count:<10} {'N/A':<12} {status:<15}")

    # Overall statistics
    if successful_tests > 0:
        overall_accuracy = (total_actual / total_expected) * 100
        print(f"{'-'*18} {'-'*15} {'-'*10} {'-'*10} {'-'*12} {'-'*15}")
        print(f"{'TOTAL':<18} {'ALL':<15} {total_expected:<10} {total_actual:<10} {overall_accuracy:.1f}{'%':<11} {'OVERALL':<15}")

        print(f"\n🎯 FINAL PERFORMANCE ANALYSIS:")
        print(f"   📊 Total birds expected: {total_expected}")
        print(f"   🐦 Total birds detected: {total_actual}")
        print(f"   📈 Overall accuracy: {overall_accuracy:.1f}%")
        print(f"   📉 Total difference: {total_actual - total_expected:+d} birds")

        # Performance rating
        if overall_accuracy >= 90:
            print(f"   🏆 Overall Rating: OUTSTANDING! 🥇")
        elif overall_accuracy >= 85:
            print(f"   ⭐ Overall Rating: EXCELLENT! ✨")
        elif overall_accuracy >= 75:
            print(f"   ✅ Overall Rating: GOOD")
        else:
            print(f"   ⚠️ Overall Rating: NEEDS IMPROVEMENT")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


final

In [ ]:
# ============================================================================
# MAIN EXECUTION
# ============================================================================

def run_all_tests():
    """Execute complete test suite"""
    print("🚀 STARTING BIRD CLASSIFICATION TESTING")
    print(f"📺 Video Source: {VIDEO_URL}")
    print(f"🧪 Testing {len(segments_to_test)} segments")
    print("=" * 80)

    # Run all tests
    all_test_results = []

    for i, segment in enumerate(segments_to_test, 1):
        print(f"\n🔬 TEST {i}/{len(segments_to_test)}")
        result = run_segment_test(segment, VIDEO_URL)
        all_test_results.append(result)

        # Show immediate results
        if result['success']:
            print(f"✅ {segment['name']}: {result['model_count']} birds detected")
            if result['accuracy_info'] and 'expected' in result['accuracy_info']:
                acc = result['accuracy_info']['accuracy_percentage']
                print(f"   Accuracy: {acc:.1f}% ({result['accuracy_info']['status']})")
        else:
            print(f"❌ {segment['name']}: Test failed")

    # Display comprehensive results
    display_comprehensive_summary(all_test_results)

    # Save master results
    master_results = {
        'test_metadata': {
            'test_date': str(datetime.now()),
            'video_url': VIDEO_URL,
            'total_segments': len(segments_to_test),
            'tracker_version': 'simplified_multiple_bird_v1'
        },
        'segment_configuration': segments_to_test,
        'test_results': all_test_results
    }

    master_results_path = f"{PATH}/data/master_test_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(master_results_path, 'w') as f:
        json.dump(master_results, f, indent=4)

    print(f"\n💾 Master results saved: {master_results_path}")
    print(f"🎉 COMPLETE TEST SUITE FINISHED!")

    return all_test_results

# ============================================================================
# USAGE EXAMPLES
# ============================================================================

print("✅ Cleaned Bird Classification Pipeline Loaded!")
print("\n🎯 USAGE:")
print("1. run_all_tests() - Execute complete test suite")
print("2. run_segment_test(segments_to_test[0], VIDEO_URL) - Test single segment")
print("\n💡 KEY IMPROVEMENTS:")
print("   • Video caching (no re-downloads)")
print("   • Simplified tracking logic")
print("   • Optimized for multiple birds")
print("   • Removed ~60% of unused code")
print("   • Clear function structure")

# Uncomment to run automatically:
# run_all_tests()